# 04C1 — Production USGS 3DEP Terrain Features

**Build ID: `04C1_3DEP_TERRAIN_PRODUCTION_V1`**

This notebook freezes the production terrain architecture after 04C0.

## 04C0 result

The pilot established:

- 304,143 operational clean-anchor fireline units;
- 381 incidents;
- 160 unique centroid 1° DEM tiles;
- 163 conservative buffered tiles;
- all representative current USGS 3DEP URLs returned HTTP 200;
- all pilot patches were ≥95% valid;
- elevation, slope, relief, and TPI were plausible;
- median naïve remote runtime was 0.41 s/unit.

The only failed pilot check was caused by applying the numerical approach azimuth even when `approach_direction_reliable=False`.

## Frozen correction

Directional terrain features are now valid only when:

\[
\boxed{
\texttt{approach\_direction\_reliable=True}
\land
\text{finite approach azimuth}
}
\]

If direction is unreliable:

- elevation remains valid;
- generic slope remains valid;
- aspect remains valid;
- terrain context remains valid;
- approach-aligned slope = missing;
- cross-approach slope = missing.

The target and fire-approach reconstruction are unchanged.

---

# Production spatial architecture

A per-unit remote DEM request would be wasteful.

Production uses globally aligned EPSG:5070 blocks:

\[
10\,\mathrm{km}\times10\,\mathrm{km}
\]

with an **800 m halo**.

Why 800 m?

- corrected local units are ≤250 m;
- context extends ±500 m around the unit centroid;
- terrain derivatives need edge padding.

Each operational unit belongs to exactly one block by its centroid.

For each block:

1. read one local 3DEP patch from the current staged COG source;
2. reproject to EPSG:5070 at 10 m;
3. calculate terrain derivatives once;
4. evaluate all fireline sample points in the block;
5. evaluate all unit centroids;
6. write one resumable block shard.

No whole-CONUS DEM download is required.

---

# Production feature set

## Along the corrected local fireline

Using the accepted 03L sample points and arc-length weights:

- elevation mean / p10 / median / p90;
- slope mean / p90 / sample maximum;
- aspect sine/cosine mean;
- signed approach-slope mean / p10 / median / p90;
- signed cross-slope tangent mean.

## Context around unit centroid

Efficient square metric windows with half-width:

- ±100 m;
- ±250 m;
- ±500 m.

For each:

- elevation mean;
- elevation standard deviation;
- local relief;
- slope mean;
- slope maximum;
- TPI-like center elevation minus neighborhood mean.

The window geometry is deliberately called out as **square metric context**, not a circular radius.

---

# Source

USGS 3DEP seamless 1/3 arc-second DEM (~10 m), current staged GeoTIFF/COG.

Terrain is treated as static bare-earth topography.

No target or future-fire information enters the terrain source.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04C1_3DEP_TERRAIN_PRODUCTION_V1"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

CORRECTED_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fireline_corrected_local_250m_10m"
)

UNIT_ROOT = CORRECTED_ROOT / "units"
SAMPLE_ROOT = CORRECTED_ROOT / "sample_points"

BASE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "hrrr_production_v2"
    / "hrrr_operational_forecast_features_v2_clean.parquet"
)

PROCESSED_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "terrain_3dep_production_v1"
)

BLOCK_SAMPLE_ROOT = (
    PROCESSED_ROOT
    / "block_samples"
)

BLOCK_SHARD_ROOT = (
    PROCESSED_ROOT
    / "block_feature_shards"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "terrain_3dep_production_v1"
)

for path in [
    PROCESSED_ROOT,
    BLOCK_SAMPLE_ROOT,
    BLOCK_SHARD_ROOT,
    OUTPUT_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

assert BASE_PATH.exists(), BASE_PATH
assert UNIT_ROOT.exists(), UNIT_ROOT
assert SAMPLE_ROOT.exists(), SAMPLE_ROOT

CRS_METRIC = "EPSG:5070"
CRS_WGS84 = "EPSG:4326"

TARGET_RESOLUTION_M = 10.0

BLOCK_SIZE_M = 10_000.0
BLOCK_HALO_M = 800.0

CONTEXT_HALF_WIDTHS_M = [
    100.0,
    250.0,
    500.0,
]

STAGED_ROOT = (
    "https://prd-tnm.s3.amazonaws.com"
    "/StagedProducts/Elevation/13/TIFF/current"
)

RESUME = True

print("=" * 96)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 96)
print("Base:", BASE_PATH)
print("Output:", PROCESSED_ROOT)

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "geopandas": "geopandas",
    "pyarrow": "pyarrow",
    "shapely": "shapely",
    "rasterio": "rasterio",
    "pyproj": "pyproj",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

import gc
import math
import time
import traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from pyproj import Transformer
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.vrt import WarpedVRT
from rasterio.transform import rowcol
from scipy.ndimage import (
    distance_transform_edt,
    uniform_filter,
    minimum_filter,
    maximum_filter,
)
from tqdm.auto import tqdm

pd.set_option(
    "display.max_columns",
    500,
)

pd.set_option(
    "display.max_rows",
    300,
)

pd.set_option(
    "display.width",
    300,
)

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)
print("Rasterio:", rasterio.__version__)

## 2. Load operational unit inventory

In [ ]:
base = pd.read_parquet(
    BASE_PATH
)

assert base[
    "unit_uid"
].is_unique

required = [
    "unit_uid",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "hold_label",
    "longitude",
    "latitude",
    "approach_azimuth_deg",
    "approach_direction_reliable",
]

missing = [
    column
    for column
    in required
    if column
    not in base.columns
]

if missing:

    raise RuntimeError(
        f"Base table missing columns: {missing}"
    )

base[
    "terrain_direction_valid"
] = (
    base[
        "approach_direction_reliable"
    ].fillna(
        False
    )
    & np.isfinite(
        pd.to_numeric(
            base[
                "approach_azimuth_deg"
            ],
            errors="coerce",
        )
    )
)

print(
    "Operational units:",
    f"{len(base):,}",
)

print(
    "Incidents:",
    base[
        "_irwin_norm"
    ].nunique(),
)

print(
    "Terrain-direction valid:",
    f"{int(base['terrain_direction_valid'].sum()):,}",
    f"({base['terrain_direction_valid'].mean()*100:.3f}%)",
)

## 3. EPSG:5070 centroid coordinates and global block assignment

In [ ]:
TO_METRIC = Transformer.from_crs(
    CRS_WGS84,
    CRS_METRIC,
    always_xy=True,
)

x_metric, y_metric = TO_METRIC.transform(
    base[
        "longitude"
    ].to_numpy(
        float
    ),
    base[
        "latitude"
    ].to_numpy(
        float
    ),
)

base[
    "terrain_centroid_x_m"
] = np.asarray(
    x_metric,
    dtype=float,
)

base[
    "terrain_centroid_y_m"
] = np.asarray(
    y_metric,
    dtype=float,
)

base[
    "terrain_block_ix"
] = np.floor(
    base[
        "terrain_centroid_x_m"
    ]
    / BLOCK_SIZE_M
).astype(
    int
)

base[
    "terrain_block_iy"
] = np.floor(
    base[
        "terrain_centroid_y_m"
    ]
    / BLOCK_SIZE_M
).astype(
    int
)

base[
    "terrain_block_id"
] = [
    f"x{ix:+07d}_y{iy:+07d}"
    for ix, iy
    in zip(
        base[
            "terrain_block_ix"
        ],
        base[
            "terrain_block_iy"
        ],
    )
]

block_manifest = (
    base.groupby(
        [
            "terrain_block_id",
            "terrain_block_ix",
            "terrain_block_iy",
        ],
        as_index=False,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        direction_valid_units=(
            "terrain_direction_valid",
            "sum",
        ),
    )
)

block_manifest[
    "core_minx"
] = (
    block_manifest[
        "terrain_block_ix"
    ]
    * BLOCK_SIZE_M
)

block_manifest[
    "core_miny"
] = (
    block_manifest[
        "terrain_block_iy"
    ]
    * BLOCK_SIZE_M
)

block_manifest[
    "core_maxx"
] = (
    block_manifest[
        "core_minx"
    ]
    + BLOCK_SIZE_M
)

block_manifest[
    "core_maxy"
] = (
    block_manifest[
        "core_miny"
    ]
    + BLOCK_SIZE_M
)

block_manifest = block_manifest.sort_values(
    "terrain_block_id"
).reset_index(
    drop=True
)

print(
    "Terrain blocks:",
    f"{len(block_manifest):,}",
)

print(
    "Median units/block:",
    block_manifest[
        "units"
    ].median(),
)

print(
    "P90 units/block:",
    block_manifest[
        "units"
    ].quantile(
        .90
    ),
)

print(
    "P99 units/block:",
    block_manifest[
        "units"
    ].quantile(
        .99
    ),
)

print(
    "Maximum units/block:",
    f"{block_manifest['units'].max():,}",
)

display(
    block_manifest.sort_values(
        "units",
        ascending=False,
    ).head(
        20
    )
)

UNIT_INDEX_PATH = (
    PROCESSED_ROOT
    / "terrain_operational_unit_index_v1.parquet"
)

MANIFEST_PATH = (
    OUTPUT_ROOT
    / "TERRAIN_BLOCK_MANIFEST.csv"
)

base[
    [
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "_engagement_norm",
        "hold_label",
        "longitude",
        "latitude",
        "approach_azimuth_deg",
        "approach_direction_reliable",
        "terrain_direction_valid",
        "terrain_centroid_x_m",
        "terrain_centroid_y_m",
        "terrain_block_id",
        "terrain_block_ix",
        "terrain_block_iy",
    ]
].to_parquet(
    UNIT_INDEX_PATH,
    index=False,
)

block_manifest.to_csv(
    MANIFEST_PATH,
    index=False,
)

## 4. Corrected unit/sample readers and arc-length weights

In [ ]:
def norm_irwin(
    series,
):
    return (
        series.astype(
            "string"
        )
        .str.strip()
        .str.replace(
            "{",
            "",
            regex=False,
        )
        .str.replace(
            "}",
            "",
            regex=False,
        )
        .str.replace(
            "-",
            "",
            regex=False,
        )
        .str.lower()
        .replace(
            {
                "<na>": pd.NA,
                "nan": pd.NA,
                "none": pd.NA,
                "": pd.NA,
            }
        )
    )


def resolve_shard(
    root,
    raw_irwin,
):
    direct = (
        root
        / f"{raw_irwin}.parquet"
    )

    if direct.exists():
        return direct

    target = norm_irwin(
        pd.Series(
            [
                raw_irwin
            ]
        )
    ).iloc[
        0
    ]

    matches = []

    for path in root.glob(
        "*.parquet"
    ):

        candidate = norm_irwin(
            pd.Series(
                [
                    path.stem
                ]
            )
        ).iloc[
            0
        ]

        if candidate == target:

            matches.append(
                path
            )

    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"Could not uniquely resolve {raw_irwin} under {root}; "
            f"matches={len(matches)}"
        )

    return matches[
        0
    ]


def read_units(
    path,
):
    df = pd.read_parquet(
        path
    )

    geometry = gpd.GeoSeries.from_wkb(
        df[
            "geometry"
        ],
        crs=CRS_METRIC,
    )

    return gpd.GeoDataFrame(
        df.drop(
            columns=[
                "geometry"
            ]
        ),
        geometry=geometry,
        crs=CRS_METRIC,
    )


def weighted_samples_for_incident(
    raw_irwin,
    wanted_unit_uids,
):
    units = read_units(
        resolve_shard(
            UNIT_ROOT,
            raw_irwin,
        )
    )

    samples = pd.read_parquet(
        resolve_shard(
            SAMPLE_ROOT,
            raw_irwin,
        )
    )

    units = units[
        units[
            "unit_uid"
        ].isin(
            wanted_unit_uids
        )
    ].copy()

    samples = samples[
        samples[
            "unit_uid"
        ].isin(
            wanted_unit_uids
        )
    ].copy()

    exploded = (
        units[
            [
                "unit_row_index",
                "geometry",
            ]
        ]
        .set_index(
            "unit_row_index"
        )
        .explode(
            index_parts=True
        )
        .reset_index()
        .rename(
            columns={
                "level_1": "part_index"
            }
        )
    )

    exploded[
        "part_index"
    ] = exploded[
        "part_index"
    ].astype(
        int
    )

    exploded[
        "part_length_m"
    ] = exploded.geometry.length.astype(
        float
    )

    counts = (
        samples.groupby(
            [
                "unit_row_index",
                "part_index",
            ],
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size": "samples_in_part"
            }
        )
    )

    samples = (
        samples.merge(
            exploded[
                [
                    "unit_row_index",
                    "part_index",
                    "part_length_m",
                ]
            ],
            on=[
                "unit_row_index",
                "part_index",
            ],
            how="left",
            validate="many_to_one",
        )
        .merge(
            counts,
            on=[
                "unit_row_index",
                "part_index",
            ],
            how="left",
            validate="many_to_one",
        )
    )

    samples[
        "sample_weight_m"
    ] = (
        samples[
            "part_length_m"
        ]
        / samples[
            "samples_in_part"
        ]
    )

    if samples[
        "sample_weight_m"
    ].isna().any():

        raise RuntimeError(
            f"NaN terrain sample weights in {raw_irwin}"
        )

    return samples[
        [
            "unit_uid",
            "x_m",
            "y_m",
            "sample_weight_m",
        ]
    ]

## 5. Build reusable block-sample shards

This is a one-time local preprocessing step.

It does **not** access 3DEP.

Operational 03L line samples are assigned to the same block as their parent unit. The resulting block sample Parquets make the remote terrain stage resumable and avoid repeatedly opening incident sample archives.

In [ ]:
BLOCK_SAMPLE_BUILD_MARKER = (
    PROCESSED_ROOT
    / "BLOCK_SAMPLE_BUILD_COMPLETE.txt"
)

if BLOCK_SAMPLE_BUILD_MARKER.exists():

    print(
        "Block sample cache already complete."
    )

else:

    temporary_frames = []

    incident_groups = list(
        base.groupby(
            "_irwin_norm",
            sort=False,
        )
    )

    for raw_irwin, group in tqdm(
        incident_groups,
        desc="Build operational terrain samples",
    ):

        wanted = set(
            group[
                "unit_uid"
            ]
        )

        samples = weighted_samples_for_incident(
            raw_irwin,
            wanted,
        )

        lookup = group[
            [
                "unit_uid",
                "terrain_block_id",
            ]
        ]

        samples = samples.merge(
            lookup,
            on="unit_uid",
            how="left",
            validate="many_to_one",
        )

        temporary_frames.append(
            samples
        )

    all_samples = pd.concat(
        temporary_frames,
        ignore_index=True,
    )

    del temporary_frames

    gc.collect()

    if all_samples[
        "terrain_block_id"
    ].isna().any():

        raise RuntimeError(
            "Some terrain samples have no block assignment."
        )

    print(
        "Operational terrain sample rows:",
        f"{len(all_samples):,}",
    )

    print(
        "Blocks with samples:",
        all_samples[
            "terrain_block_id"
        ].nunique(),
    )

    for block_id, group in tqdm(
        all_samples.groupby(
            "terrain_block_id",
            sort=False,
        ),
        desc="Write block sample shards",
    ):

        group[
            [
                "unit_uid",
                "x_m",
                "y_m",
                "sample_weight_m",
            ]
        ].to_parquet(
            BLOCK_SAMPLE_ROOT
            / f"{block_id}.parquet",
            index=False,
        )

    del all_samples

    gc.collect()

    BLOCK_SAMPLE_BUILD_MARKER.write_text(
        (
            f"{NOTEBOOK_BUILD_ID}\n"
            f"completed_utc={datetime.now(timezone.utc).isoformat()}\n"
        ),
        encoding="utf-8",
    )

    print(
        "Block sample cache complete."
    )

## 6. USGS tile and local DEM helpers

In [ ]:
TO_WGS84 = Transformer.from_crs(
    CRS_METRIC,
    CRS_WGS84,
    always_xy=True,
)


def tile_id_from_cell(
    lat_cell_south,
    lon_cell_west,
):
    north_edge = int(
        lat_cell_south
        + 1
    )

    ns = (
        f"n{abs(north_edge):02d}"
        if north_edge >= 0
        else f"s{abs(north_edge):02d}"
    )

    ew = (
        f"w{abs(int(lon_cell_west)):03d}"
        if lon_cell_west < 0
        else f"e{abs(int(lon_cell_west)):03d}"
    )

    return ns + ew


def tile_ids_for_bbox(
    min_lon,
    min_lat,
    max_lon,
    max_lat,
):
    lon0 = math.floor(
        float(
            min_lon
        )
    )

    lon1 = math.floor(
        float(
            max_lon
        )
        - 1e-12
    )

    lat0 = math.floor(
        float(
            min_lat
        )
    )

    lat1 = math.floor(
        float(
            max_lat
        )
        - 1e-12
    )

    return sorted(
        {
            tile_id_from_cell(
                y,
                x,
            )
            for y
            in range(
                lat0,
                lat1 + 1,
            )
            for x
            in range(
                lon0,
                lon1 + 1,
            )
        }
    )


def tile_url(
    tile_id,
):
    return (
        f"{STAGED_ROOT}/"
        f"{tile_id}/"
        f"USGS_13_{tile_id}.tif"
    )


def metric_bounds_to_wgs84(
    bounds,
):
    minx, miny, maxx, maxy = bounds

    corners = [
        TO_WGS84.transform(
            minx,
            miny,
        ),
        TO_WGS84.transform(
            minx,
            maxy,
        ),
        TO_WGS84.transform(
            maxx,
            miny,
        ),
        TO_WGS84.transform(
            maxx,
            maxy,
        ),
    ]

    lons = [
        point[
            0
        ]
        for point
        in corners
    ]

    lats = [
        point[
            1
        ]
        for point
        in corners
    ]

    return (
        min(
            lons
        ),
        min(
            lats
        ),
        max(
            lons
        ),
        max(
            lats
        ),
    )


def read_dem_patch(
    bounds_5070,
):
    bbox = metric_bounds_to_wgs84(
        bounds_5070
    )

    ids = tile_ids_for_bbox(
        *bbox
    )

    datasets = []
    vrts = []

    try:

        for tile_id in ids:

            ds = rasterio.open(
                tile_url(
                    tile_id
                )
            )

            datasets.append(
                ds
            )

            vrts.append(
                WarpedVRT(
                    ds,
                    crs=CRS_METRIC,
                    resampling=Resampling.bilinear,
                    nodata=np.nan,
                )
            )

        mosaic, transform = merge(
            vrts,
            bounds=bounds_5070,
            res=TARGET_RESOLUTION_M,
            nodata=np.nan,
            dtype="float32",
            method="first",
        )

        dem = mosaic[
            0
        ].astype(
            np.float64
        )

        dem[
            ~np.isfinite(
                dem
            )
        ] = np.nan

        dem[
            np.abs(
                dem
            )
            > 100000
        ] = np.nan

        return (
            dem,
            transform,
            ids,
        )

    finally:

        for obj in vrts:

            try:
                obj.close()
            except Exception:
                pass

        for obj in datasets:

            try:
                obj.close()
            except Exception:
                pass

## 7. Terrain derivatives and NaN-aware context filters

In [ ]:
def nearest_fill(
    array,
):
    valid = np.isfinite(
        array
    )

    if not valid.any():

        raise RuntimeError(
            "DEM patch contains no finite values."
        )

    filled = array.copy()

    if not valid.all():

        indices = distance_transform_edt(
            ~valid,
            return_distances=False,
            return_indices=True,
        )

        filled[
            ~valid
        ] = filled[
            tuple(
                idx[
                    ~valid
                ]
                for idx
                in indices
            )
        ]

    return filled, valid


def terrain_derivatives(
    dem,
):
    filled, valid = nearest_fill(
        dem
    )

    grad_row, grad_col = np.gradient(
        filled,
        TARGET_RESOLUTION_M,
        TARGET_RESOLUTION_M,
    )

    dzdx = grad_col

    # Raster rows increase southward.
    dzdy = -grad_row

    slope_tangent = np.hypot(
        dzdx,
        dzdy,
    )

    slope_deg = np.degrees(
        np.arctan(
            slope_tangent
        )
    )

    aspect_deg = (
        np.degrees(
            np.arctan2(
                -dzdx,
                -dzdy,
            )
        )
        + 360.0
    ) % 360.0

    aspect_deg[
        slope_tangent
        < 1e-10
    ] = np.nan

    for array in [
        dzdx,
        dzdy,
        slope_deg,
        aspect_deg,
    ]:

        array[
            ~valid
        ] = np.nan

    return (
        dzdx,
        dzdy,
        slope_deg,
        aspect_deg,
    )


def nan_box_mean_std(
    array,
    size,
):
    valid = np.isfinite(
        array
    )

    values = np.where(
        valid,
        array,
        0.0,
    )

    weights = valid.astype(
        float
    )

    # uniform_filter returns local mean;
    # dividing filtered values by filtered valid fraction
    # yields a NaN-aware local mean.
    filtered_value = uniform_filter(
        values,
        size=size,
        mode="nearest",
    )

    filtered_weight = uniform_filter(
        weights,
        size=size,
        mode="nearest",
    )

    mean = np.divide(
        filtered_value,
        filtered_weight,
        out=np.full_like(
            filtered_value,
            np.nan,
            dtype=float,
        ),
        where=filtered_weight > 0,
    )

    filtered_sq = uniform_filter(
        values**2,
        size=size,
        mode="nearest",
    )

    mean_sq = np.divide(
        filtered_sq,
        filtered_weight,
        out=np.full_like(
            filtered_sq,
            np.nan,
            dtype=float,
        ),
        where=filtered_weight > 0,
    )

    variance = np.maximum(
        mean_sq
        - mean**2,
        0.0,
    )

    std = np.sqrt(
        variance
    )

    return (
        mean,
        std,
        filtered_weight,
    )


def nan_box_minmax(
    array,
    size,
):
    valid = np.isfinite(
        array
    )

    minimum_input = np.where(
        valid,
        array,
        np.inf,
    )

    maximum_input = np.where(
        valid,
        array,
        -np.inf,
    )

    local_min = minimum_filter(
        minimum_input,
        size=size,
        mode="nearest",
    )

    local_max = maximum_filter(
        maximum_input,
        size=size,
        mode="nearest",
    )

    local_min[
        ~np.isfinite(
            local_min
        )
    ] = np.nan

    local_max[
        ~np.isfinite(
            local_max
        )
    ] = np.nan

    return (
        local_min,
        local_max,
    )


def build_context_rasters(
    dem,
    slope_deg,
):
    outputs = {}

    for half_width in CONTEXT_HALF_WIDTHS_M:

        radius_pixels = int(
            round(
                half_width
                / TARGET_RESOLUTION_M
            )
        )

        size = (
            2
            * radius_pixels
            + 1
        )

        elev_mean, elev_std, valid_fraction = (
            nan_box_mean_std(
                dem,
                size,
            )
        )

        elev_min, elev_max = nan_box_minmax(
            dem,
            size,
        )

        slope_mean, _, _ = nan_box_mean_std(
            slope_deg,
            size,
        )

        _, slope_max = nan_box_minmax(
            slope_deg,
            size,
        )

        tag = int(
            half_width
        )

        outputs[
            tag
        ] = {
            "elevation_mean": elev_mean,
            "elevation_std": elev_std,
            "relief": (
                elev_max
                - elev_min
            ),
            "slope_mean": slope_mean,
            "slope_max": slope_max,
            "valid_fraction": valid_fraction,
        }

    return outputs

## 8. Raster sampling and weighted aggregation

In [ ]:
def values_at_xy(
    array,
    transform,
    xs,
    ys,
):
    rows, cols = rowcol(
        transform,
        np.asarray(
            xs,
            dtype=float,
        ),
        np.asarray(
            ys,
            dtype=float,
        ),
    )

    rows = np.asarray(
        rows,
        dtype=int,
    )

    cols = np.asarray(
        cols,
        dtype=int,
    )

    good = (
        (
            rows
            >= 0
        )
        & (
            rows
            < array.shape[
                0
            ]
        )
        & (
            cols
            >= 0
        )
        & (
            cols
            < array.shape[
                1
            ]
        )
    )

    result = np.full(
        len(
            rows
        ),
        np.nan,
        dtype=float,
    )

    result[
        good
    ] = array[
        rows[
            good
        ],
        cols[
            good
        ],
    ]

    return result


def weighted_mean(
    values,
    weights,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    good = (
        np.isfinite(
            values
        )
        & np.isfinite(
            weights
        )
        & (
            weights
            > 0
        )
    )

    if not good.any():

        return np.nan

    return float(
        np.sum(
            values[
                good
            ]
            * weights[
                good
            ]
        )
        / np.sum(
            weights[
                good
            ]
        )
    )


def weighted_quantile(
    values,
    weights,
    q,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    good = (
        np.isfinite(
            values
        )
        & np.isfinite(
            weights
        )
        & (
            weights
            > 0
        )
    )

    if not good.any():

        return np.nan

    values = values[
        good
    ]

    weights = weights[
        good
    ]

    order = np.argsort(
        values
    )

    values = values[
        order
    ]

    weights = weights[
        order
    ]

    cumulative = np.cumsum(
        weights
    )

    target = (
        float(
            q
        )
        * cumulative[
            -1
        ]
    )

    index = int(
        np.searchsorted(
            cumulative,
            target,
            side="left",
        )
    )

    return float(
        values[
            min(
                index,
                len(
                    values
                )
                - 1,
            )
        ]
    )

## 9. Per-block feature extractor

In [ ]:
def process_block(
    block_row,
):
    block_id = block_row[
        "terrain_block_id"
    ]

    units = base[
        base[
            "terrain_block_id"
        ]
        == block_id
    ].copy().reset_index(
        drop=True
    )

    samples_path = (
        BLOCK_SAMPLE_ROOT
        / f"{block_id}.parquet"
    )

    if not samples_path.exists():

        raise RuntimeError(
            f"Missing block sample shard: {samples_path}"
        )

    samples = pd.read_parquet(
        samples_path
    )

    bounds = (
        float(
            block_row[
                "core_minx"
            ]
        )
        - BLOCK_HALO_M,
        float(
            block_row[
                "core_miny"
            ]
        )
        - BLOCK_HALO_M,
        float(
            block_row[
                "core_maxx"
            ]
        )
        + BLOCK_HALO_M,
        float(
            block_row[
                "core_maxy"
            ]
        )
        + BLOCK_HALO_M,
    )

    start = time.perf_counter()

    dem, transform, tile_ids = (
        read_dem_patch(
            bounds
        )
    )

    dem_valid_fraction = float(
        np.isfinite(
            dem
        ).mean()
    )

    dzdx, dzdy, slope_deg, aspect_deg = (
        terrain_derivatives(
            dem
        )
    )

    context = build_context_rasters(
        dem,
        slope_deg,
    )

    # ---------------------------------------------------------
    # Sample terrain once for every line sample in this block.
    # ---------------------------------------------------------

    sx = samples[
        "x_m"
    ].to_numpy(
        float
    )

    sy = samples[
        "y_m"
    ].to_numpy(
        float
    )

    samples[
        "_elevation"
    ] = values_at_xy(
        dem,
        transform,
        sx,
        sy,
    )

    samples[
        "_slope_deg"
    ] = values_at_xy(
        slope_deg,
        transform,
        sx,
        sy,
    )

    samples[
        "_aspect_deg"
    ] = values_at_xy(
        aspect_deg,
        transform,
        sx,
        sy,
    )

    samples[
        "_dzdx"
    ] = values_at_xy(
        dzdx,
        transform,
        sx,
        sy,
    )

    samples[
        "_dzdy"
    ] = values_at_xy(
        dzdy,
        transform,
        sx,
        sy,
    )

    direction_lookup = units[
        [
            "unit_uid",
            "approach_azimuth_deg",
            "terrain_direction_valid",
        ]
    ]

    samples = samples.merge(
        direction_lookup,
        on="unit_uid",
        how="left",
        validate="many_to_one",
    )

    theta = np.deg2rad(
        pd.to_numeric(
            samples[
                "approach_azimuth_deg"
            ],
            errors="coerce",
        ).to_numpy(
            float
        )
    )

    valid_direction = samples[
        "terrain_direction_valid"
    ].fillna(
        False
    ).to_numpy(
        bool
    )

    approach_east = np.sin(
        theta
    )

    approach_north = np.cos(
        theta
    )

    cross_east = np.cos(
        theta
    )

    cross_north = -np.sin(
        theta
    )

    along_tangent = (
        samples[
            "_dzdx"
        ].to_numpy(
            float
        )
        * approach_east
        + samples[
            "_dzdy"
        ].to_numpy(
            float
        )
        * approach_north
    )

    cross_tangent = (
        samples[
            "_dzdx"
        ].to_numpy(
            float
        )
        * cross_east
        + samples[
            "_dzdy"
        ].to_numpy(
            float
        )
        * cross_north
    )

    along_tangent[
        ~valid_direction
    ] = np.nan

    cross_tangent[
        ~valid_direction
    ] = np.nan

    samples[
        "_approach_slope_deg"
    ] = np.degrees(
        np.arctan(
            along_tangent
        )
    )

    samples[
        "_cross_slope_tangent"
    ] = cross_tangent

    aspect_rad = np.deg2rad(
        samples[
            "_aspect_deg"
        ].to_numpy(
            float
        )
    )

    samples[
        "_aspect_sin"
    ] = np.sin(
        aspect_rad
    )

    samples[
        "_aspect_cos"
    ] = np.cos(
        aspect_rad
    )

    rows = []

    for unit_uid, group in samples.groupby(
        "unit_uid",
        sort=False,
    ):

        weights = group[
            "sample_weight_m"
        ].to_numpy(
            float
        )

        row = {
            "unit_uid": unit_uid,
            "terrain_elevation_mean_m": weighted_mean(
                group[
                    "_elevation"
                ],
                weights,
            ),
            "terrain_elevation_p10_m": weighted_quantile(
                group[
                    "_elevation"
                ],
                weights,
                .10,
            ),
            "terrain_elevation_median_m": weighted_quantile(
                group[
                    "_elevation"
                ],
                weights,
                .50,
            ),
            "terrain_elevation_p90_m": weighted_quantile(
                group[
                    "_elevation"
                ],
                weights,
                .90,
            ),
            "terrain_slope_mean_deg": weighted_mean(
                group[
                    "_slope_deg"
                ],
                weights,
            ),
            "terrain_slope_p90_deg": weighted_quantile(
                group[
                    "_slope_deg"
                ],
                weights,
                .90,
            ),
            "terrain_slope_max_sample_deg": (
                float(
                    np.nanmax(
                        group[
                            "_slope_deg"
                        ]
                    )
                )
                if np.isfinite(
                    group[
                        "_slope_deg"
                    ]
                ).any()
                else np.nan
            ),
            "terrain_aspect_sin_mean": weighted_mean(
                group[
                    "_aspect_sin"
                ],
                weights,
            ),
            "terrain_aspect_cos_mean": weighted_mean(
                group[
                    "_aspect_cos"
                ],
                weights,
            ),
            "terrain_approach_slope_mean_deg": weighted_mean(
                group[
                    "_approach_slope_deg"
                ],
                weights,
            ),
            "terrain_approach_slope_p10_deg": weighted_quantile(
                group[
                    "_approach_slope_deg"
                ],
                weights,
                .10,
            ),
            "terrain_approach_slope_median_deg": weighted_quantile(
                group[
                    "_approach_slope_deg"
                ],
                weights,
                .50,
            ),
            "terrain_approach_slope_p90_deg": weighted_quantile(
                group[
                    "_approach_slope_deg"
                ],
                weights,
                .90,
            ),
            "terrain_cross_slope_tangent_mean": weighted_mean(
                group[
                    "_cross_slope_tangent"
                ],
                weights,
            ),
            "terrain_line_valid_fraction": float(
                np.average(
                    np.isfinite(
                        group[
                            "_elevation"
                        ]
                    ).astype(
                        float
                    ),
                    weights=weights,
                )
            ),
        }

        rows.append(
            row
        )

    line_features = pd.DataFrame(
        rows
    )

    # ---------------------------------------------------------
    # Context features at all unit centroids.
    # ---------------------------------------------------------

    cx = units[
        "terrain_centroid_x_m"
    ].to_numpy(
        float
    )

    cy = units[
        "terrain_centroid_y_m"
    ].to_numpy(
        float
    )

    centroid_features = pd.DataFrame(
        {
            "unit_uid": units[
                "unit_uid"
            ].to_numpy(),
            "terrain_centroid_elevation_m": values_at_xy(
                dem,
                transform,
                cx,
                cy,
            ),
        }
    )

    for half_width in CONTEXT_HALF_WIDTHS_M:

        tag = int(
            half_width
        )

        context_arrays = context[
            tag
        ]

        elev_mean = values_at_xy(
            context_arrays[
                "elevation_mean"
            ],
            transform,
            cx,
            cy,
        )

        centroid_features[
            f"terrain_context_{tag}m_elevation_mean_m"
        ] = elev_mean

        centroid_features[
            f"terrain_context_{tag}m_elevation_std_m"
        ] = values_at_xy(
            context_arrays[
                "elevation_std"
            ],
            transform,
            cx,
            cy,
        )

        centroid_features[
            f"terrain_context_{tag}m_relief_m"
        ] = values_at_xy(
            context_arrays[
                "relief"
            ],
            transform,
            cx,
            cy,
        )

        centroid_features[
            f"terrain_context_{tag}m_slope_mean_deg"
        ] = values_at_xy(
            context_arrays[
                "slope_mean"
            ],
            transform,
            cx,
            cy,
        )

        centroid_features[
            f"terrain_context_{tag}m_slope_max_deg"
        ] = values_at_xy(
            context_arrays[
                "slope_max"
            ],
            transform,
            cx,
            cy,
        )

        centroid_features[
            f"terrain_context_{tag}m_valid_fraction"
        ] = values_at_xy(
            context_arrays[
                "valid_fraction"
            ],
            transform,
            cx,
            cy,
        )

        centroid_features[
            f"terrain_context_{tag}m_tpi_m"
        ] = (
            centroid_features[
                "terrain_centroid_elevation_m"
            ]
            - elev_mean
        )

    output = units[
        [
            "unit_uid",
            "_irwin_norm",
            "IncidentName",
            "Year",
            "_engagement_norm",
            "hold_label",
            "approach_azimuth_deg",
            "approach_direction_reliable",
            "terrain_direction_valid",
            "terrain_block_id",
        ]
    ].merge(
        line_features,
        on="unit_uid",
        how="left",
        validate="one_to_one",
    ).merge(
        centroid_features,
        on="unit_uid",
        how="left",
        validate="one_to_one",
    )

    output[
        "terrain_dem_patch_valid_fraction"
    ] = dem_valid_fraction

    output[
        "terrain_source_tile_ids"
    ] = ";".join(
        tile_ids
    )

    output[
        "terrain_block_runtime_s"
    ] = (
        time.perf_counter()
        - start
    )

    if not output[
        "unit_uid"
    ].is_unique:

        raise RuntimeError(
            f"Duplicate unit_uid in terrain block {block_id}"
        )

    if len(
        output
    ) != len(
        units
    ):

        raise RuntimeError(
            f"Block output mismatch for {block_id}: "
            f"{len(output)} vs {len(units)}"
        )

    # Enforce the frozen directional reliability rule.
    invalid_direction = (
        ~output[
            "terrain_direction_valid"
        ].fillna(
            False
        )
    )

    directional_columns = [
        "terrain_approach_slope_mean_deg",
        "terrain_approach_slope_p10_deg",
        "terrain_approach_slope_median_deg",
        "terrain_approach_slope_p90_deg",
        "terrain_cross_slope_tangent_mean",
    ]

    if (
        output.loc[
            invalid_direction,
            directional_columns,
        ]
        .notna()
        .any()
        .any()
    ):

        raise RuntimeError(
            f"Directional terrain leaked into unreliable directions in {block_id}"
        )

    return output

## 10. One-block preflight

In [ ]:
# Choose a modest nontrivial block near the median size.
nontrivial = block_manifest[
    block_manifest[
        "units"
    ]
    >= 3
].copy()

target = nontrivial[
    "units"
].median()

preflight_row = (
    nontrivial.assign(
        delta=(
            nontrivial[
                "units"
            ]
            - target
        ).abs()
    )
    .sort_values(
        "delta"
    )
    .iloc[
        0
    ]
)

print(
    "Preflight block:",
    preflight_row[
        "terrain_block_id"
    ],
)

print(
    "Units:",
    int(
        preflight_row[
            "units"
        ]
    ),
)

start = time.perf_counter()

with rasterio.Env(
    GDAL_HTTP_MULTIRANGE="YES",
    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",
    VSI_CACHE="TRUE",
    VSI_CACHE_SIZE=128
    * 1024
    * 1024,
):

    preflight_features = process_block(
        preflight_row
    )

preflight_runtime = (
    time.perf_counter()
    - start
)

print(
    "Runtime:",
    f"{preflight_runtime:.2f} s",
)

print(
    "Direction-valid units:",
    int(
        preflight_features[
            "terrain_direction_valid"
        ].sum()
    ),
)

print(
    "Approach-slope available:",
    int(
        preflight_features[
            "terrain_approach_slope_mean_deg"
        ].notna().sum()
    ),
)

display(
    preflight_features[
        [
            "unit_uid",
            "IncidentName",
            "terrain_direction_valid",
            "terrain_elevation_mean_m",
            "terrain_slope_mean_deg",
            "terrain_approach_slope_mean_deg",
            "terrain_context_500m_relief_m",
            "terrain_context_500m_tpi_m",
        ]
    ].head(
        20
    )
)

assert (
    len(
        preflight_features
    )
    == int(
        preflight_row[
            "units"
        ]
    )
)

assert preflight_features[
    "unit_uid"
].is_unique

invalid = (
    ~preflight_features[
        "terrain_direction_valid"
    ]
)

assert (
    preflight_features.loc[
        invalid,
        "terrain_approach_slope_mean_deg",
    ]
    .isna()
    .all()
)

print(
    "04C1 BLOCK PREFLIGHT: PASS"
)

## 11. Three-block runtime benchmark

Run this before full production.

It evaluates:

- a small block;
- a median-size block;
- a ~p90-size block.

The projected runtime is intentionally rough because remote COG cache state and tile complexity vary.

If the benchmark is unexpectedly slow, stop before full production and inspect the benchmark table.

In [ ]:
def nearest_manifest_row(
    target_units,
):
    return (
        block_manifest.assign(
            delta=(
                block_manifest[
                    "units"
                ]
                - target_units
            ).abs()
        )
        .sort_values(
            "delta"
        )
        .iloc[
            0
        ]
    )


benchmark_targets = {
    "small": max(
        3,
        int(
            block_manifest[
                "units"
            ].quantile(
                .10
            )
        ),
    ),
    "medium": int(
        block_manifest[
            "units"
        ].median()
    ),
    "p90_large": int(
        block_manifest[
            "units"
        ].quantile(
            .90
        )
    ),
}

benchmark_rows = []

used_blocks = set()

with rasterio.Env(
    GDAL_HTTP_MULTIRANGE="YES",
    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",
    VSI_CACHE="TRUE",
    VSI_CACHE_SIZE=128
    * 1024
    * 1024,
):

    for label, target_units in benchmark_targets.items():

        candidate = nearest_manifest_row(
            target_units
        )

        if candidate[
            "terrain_block_id"
        ] in used_blocks:

            alternatives = (
                block_manifest[
                    ~block_manifest[
                        "terrain_block_id"
                    ].isin(
                        used_blocks
                    )
                ]
                .assign(
                    delta=lambda df: (
                        df[
                            "units"
                        ]
                        - target_units
                    ).abs()
                )
                .sort_values(
                    "delta"
                )
            )

            candidate = alternatives.iloc[
                0
            ]

        used_blocks.add(
            candidate[
                "terrain_block_id"
            ]
        )

        t0 = time.perf_counter()

        features = process_block(
            candidate
        )

        runtime = (
            time.perf_counter()
            - t0
        )

        benchmark_rows.append(
            {
                "benchmark_size": label,
                "terrain_block_id": candidate[
                    "terrain_block_id"
                ],
                "units": int(
                    candidate[
                        "units"
                    ]
                ),
                "incidents": int(
                    candidate[
                        "incidents"
                    ]
                ),
                "runtime_seconds": runtime,
                "units_per_second": (
                    len(
                        features
                    )
                    / runtime
                    if runtime
                    > 0
                    else np.nan
                ),
            }
        )

        del features

        gc.collect()

terrain_benchmark = pd.DataFrame(
    benchmark_rows
)

display(
    terrain_benchmark
)

weighted_seconds_per_block = (
    terrain_benchmark[
        "runtime_seconds"
    ].sum()
    / len(
        terrain_benchmark
    )
)

rough_projected_hours = (
    weighted_seconds_per_block
    * len(
        block_manifest
    )
    / 3600.0
)

print(
    "Rough projected full runtime:",
    f"{rough_projected_hours:.2f} h",
)

terrain_benchmark.to_csv(
    OUTPUT_ROOT
    / "TERRAIN_BLOCK_RUNTIME_BENCHMARK.csv",
    index=False,
)

## 12. Resumable block production

Only run after the block preflight and benchmark are acceptable.

One Parquet shard is written per 10-km terrain block.

In [ ]:
RUN_LOG_PATH = (
    OUTPUT_ROOT
    / "04C1_RUN_LOG.csv"
)

run_rows = []

with rasterio.Env(
    GDAL_HTTP_MULTIRANGE="YES",
    GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES",
    VSI_CACHE="TRUE",
    VSI_CACHE_SIZE=256
    * 1024
    * 1024,
):

    bar = tqdm(
        block_manifest.to_dict(
            "records"
        ),
        total=len(
            block_manifest
        ),
        desc="04C1 terrain blocks",
    )

    for block_row in bar:

        block_id = block_row[
            "terrain_block_id"
        ]

        shard_path = (
            BLOCK_SHARD_ROOT
            / f"{block_id}.parquet"
        )

        start = time.perf_counter()

        try:

            if (
                RESUME
                and shard_path.exists()
            ):

                preview = pd.read_parquet(
                    shard_path,
                    columns=[
                        "unit_uid",
                        "terrain_elevation_mean_m",
                        "terrain_approach_slope_mean_deg",
                    ],
                )

                run_rows.append(
                    {
                        "terrain_block_id": block_id,
                        "status": "SKIPPED_COMPLETE",
                        "units": len(
                            preview
                        ),
                        "runtime_seconds": (
                            time.perf_counter()
                            - start
                        ),
                    }
                )

                bar.set_postfix(
                    {
                        "block": block_id,
                        "status": "resume",
                    }
                )

                continue

            features = process_block(
                block_row
            )

            features.to_parquet(
                shard_path,
                index=False,
            )

            runtime = (
                time.perf_counter()
                - start
            )

            run_rows.append(
                {
                    "terrain_block_id": block_id,
                    "status": "OK",
                    "units": len(
                        features
                    ),
                    "incidents": int(
                        block_row[
                            "incidents"
                        ]
                    ),
                    "direction_valid_units": int(
                        features[
                            "terrain_direction_valid"
                        ].sum()
                    ),
                    "elevation_available_units": int(
                        features[
                            "terrain_elevation_mean_m"
                        ].notna().sum()
                    ),
                    "runtime_seconds": runtime,
                }
            )

            bar.set_postfix(
                {
                    "block": block_id,
                    "units": len(
                        features
                    ),
                    "sec": f"{runtime:.1f}",
                }
            )

            del features

            gc.collect()

        except Exception as exc:

            error_path = (
                OUTPUT_ROOT
                / f"ERROR_{block_id}.txt"
            )

            error_path.write_text(
                traceback.format_exc(),
                encoding="utf-8",
            )

            run_rows.append(
                {
                    "terrain_block_id": block_id,
                    "status": "ERROR",
                    "units": int(
                        block_row[
                            "units"
                        ]
                    ),
                    "error": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                    "runtime_seconds": (
                        time.perf_counter()
                        - start
                    ),
                }
            )

            bar.set_postfix(
                {
                    "block": block_id,
                    "status": "ERROR",
                }
            )

        pd.DataFrame(
            run_rows
        ).to_csv(
            RUN_LOG_PATH,
            index=False,
        )

## 13. Completion check and combine

In [ ]:
shards = sorted(
    BLOCK_SHARD_ROOT.glob(
        "*.parquet"
    )
)

print(
    "Completed block shards:",
    len(
        shards
    ),
    "/",
    len(
        block_manifest
    ),
)

if len(
    shards
) != len(
    block_manifest
):

    run_log = pd.read_csv(
        RUN_LOG_PATH
    )

    display(
        run_log[
            run_log[
                "status"
            ]
            == "ERROR"
        ]
    )

    raise RuntimeError(
        "04C1 incomplete. Fix true errors and rerun; completed block shards are resumable."
    )

frames = []

for path in tqdm(
    shards,
    desc="Combine terrain block shards",
):

    frames.append(
        pd.read_parquet(
            path
        )
    )

terrain_features = pd.concat(
    frames,
    ignore_index=True,
)

del frames

gc.collect()

assert terrain_features[
    "unit_uid"
].is_unique

assert (
    len(
        terrain_features
    )
    == len(
        base
    )
)

FEATURES_PATH = (
    PROCESSED_ROOT
    / "terrain_3dep_features_v1.parquet"
)

terrain_features.to_parquet(
    FEATURES_PATH,
    index=False,
)

print(
    "Rows:",
    f"{len(terrain_features):,}",
)

print(
    "Columns:",
    len(
        terrain_features.columns
    ),
)

print(
    "Saved:",
    FEATURES_PATH
)

## 14. Direction-reliability and coverage QA

In [ ]:
direction_qa = (
    terrain_features.groupby(
        "terrain_direction_valid",
        dropna=False,
        as_index=False,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        elevation_available=(
            "terrain_elevation_mean_m",
            lambda s: s.notna().sum(),
        ),
        approach_slope_available=(
            "terrain_approach_slope_mean_deg",
            lambda s: s.notna().sum(),
        ),
    )
)

display(
    direction_qa
)

invalid_direction = (
    ~terrain_features[
        "terrain_direction_valid"
    ].fillna(
        False
    )
)

directional_columns = [
    "terrain_approach_slope_mean_deg",
    "terrain_approach_slope_p10_deg",
    "terrain_approach_slope_median_deg",
    "terrain_approach_slope_p90_deg",
    "terrain_cross_slope_tangent_mean",
]

assert (
    terrain_features.loc[
        invalid_direction,
        directional_columns,
    ]
    .isna()
    .all()
    .all()
)

coverage_qa = (
    terrain_features.groupby(
        "_engagement_norm",
        as_index=False,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        elevation_available=(
            "terrain_elevation_mean_m",
            lambda s: s.notna().sum(),
        ),
        slope_available=(
            "terrain_slope_mean_deg",
            lambda s: s.notna().sum(),
        ),
        context500_available=(
            "terrain_context_500m_elevation_mean_m",
            lambda s: s.notna().sum(),
        ),
    )
)

for column in [
    "elevation_available",
    "slope_available",
    "context500_available",
]:

    coverage_qa[
        column
        + "_pct"
    ] = (
        coverage_qa[
            column
        ]
        / coverage_qa[
            "units"
        ]
        * 100.0
    )

display(
    coverage_qa
)

direction_qa.to_csv(
    OUTPUT_ROOT
    / "TERRAIN_DIRECTION_RELIABILITY_QA.csv",
    index=False,
)

coverage_qa.to_csv(
    OUTPUT_ROOT
    / "TERRAIN_COVERAGE_BY_CLASS.csv",
    index=False,
)

## 15. Terrain plausibility

In [ ]:
plausibility_columns = [
    "terrain_elevation_mean_m",
    "terrain_slope_mean_deg",
    "terrain_slope_p90_deg",
    "terrain_slope_max_sample_deg",
    "terrain_approach_slope_mean_deg",
    "terrain_approach_slope_p10_deg",
    "terrain_approach_slope_p90_deg",
    "terrain_context_100m_relief_m",
    "terrain_context_250m_relief_m",
    "terrain_context_500m_relief_m",
    "terrain_context_100m_tpi_m",
    "terrain_context_250m_tpi_m",
    "terrain_context_500m_tpi_m",
]

rows = []

for column in plausibility_columns:

    values = terrain_features[
        column
    ].dropna()

    rows.append(
        {
            "feature": column,
            "n": len(
                values
            ),
            "min": (
                values.min()
                if len(
                    values
                )
                else np.nan
            ),
            "p01": (
                values.quantile(
                    .01
                )
                if len(
                    values
                )
                else np.nan
            ),
            "median": (
                values.median()
                if len(
                    values
                )
                else np.nan
            ),
            "p99": (
                values.quantile(
                    .99
                )
                if len(
                    values
                )
                else np.nan
            ),
            "max": (
                values.max()
                if len(
                    values
                )
                else np.nan
            ),
        }
    )

terrain_plausibility = pd.DataFrame(
    rows
)

display(
    terrain_plausibility
)

terrain_plausibility.to_csv(
    OUTPUT_ROOT
    / "TERRAIN_PRODUCTION_PLAUSIBILITY.csv",
    index=False,
)

## 16. Missingness and runtime

In [ ]:
terrain_columns = [
    column
    for column
    in terrain_features.columns
    if column.startswith(
        "terrain_"
    )
]

missingness = pd.DataFrame(
    {
        "feature": terrain_columns,
        "missing_count": [
            terrain_features[
                column
            ].isna().sum()
            for column
            in terrain_columns
        ],
    }
)

missingness[
    "missing_pct"
] = (
    missingness[
        "missing_count"
    ]
    / len(
        terrain_features
    )
    * 100.0
)

missingness = missingness.sort_values(
    [
        "missing_pct",
        "feature",
    ],
    ascending=[
        False,
        True,
    ],
)

display(
    missingness.head(
        100
    )
)

missingness.to_csv(
    OUTPUT_ROOT
    / "TERRAIN_FEATURE_MISSINGNESS.csv",
    index=False,
)

run_log = pd.read_csv(
    RUN_LOG_PATH
)

runtime_qa = (
    run_log.groupby(
        "status",
        as_index=False,
    )
    .agg(
        blocks=(
            "terrain_block_id",
            "size",
        ),
        units=(
            "units",
            "sum",
        ),
        total_runtime_s=(
            "runtime_seconds",
            "sum",
        ),
        median_block_runtime_s=(
            "runtime_seconds",
            "median",
        ),
        p90_block_runtime_s=(
            "runtime_seconds",
            lambda s: s.quantile(
                .90
            ),
        ),
        max_block_runtime_s=(
            "runtime_seconds",
            "max",
        ),
    )
)

display(
    runtime_qa
)

runtime_qa.to_csv(
    OUTPUT_ROOT
    / "TERRAIN_RUNTIME_AUDIT.csv",
    index=False,
)

## 17. Final production report

In [ ]:
def tomd(
    frame,
):
    try:
        return frame.to_markdown(
            index=False
        )
    except Exception:
        return frame.to_string(
            index=False
        )


report = f"""# 04C1 Production USGS 3DEP Terrain Feature Audit

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Production cohort

Units: **{len(terrain_features):,}**

Incidents: **{terrain_features["_irwin_norm"].nunique():,}**

Terrain blocks: **{len(block_manifest):,}**

## Direction rule

Directional terrain is valid only when:

`approach_direction_reliable=True`

and the approach azimuth is finite.

The failed 04C0 reliability check is therefore corrected without changing the underlying fire-approach reconstruction.

## Direction QA

{tomd(direction_qa)}

## Coverage

{tomd(coverage_qa)}

## Plausibility

{tomd(terrain_plausibility)}

## Runtime

{tomd(runtime_qa)}

## Source and spatial semantics

Source: USGS 3DEP 1/3-arc-second seamless bare-earth DEM.

Remote COG windows are reprojected locally to EPSG:5070 at 10 m.

Corrected 03L fireline sample points and the accepted arc-length weights define on-line terrain summaries.

## Generic terrain

Elevation, generic slope, aspect, and context do not depend on fire-approach reliability.

## Directional terrain

Positive signed approach slope means terrain rises in the incoming fire-to-line direction.

Negative signed approach slope means terrain descends in the incoming direction.

Directional values are missing by design when the 04A approach direction is unreliable.

## Context windows

Production context uses efficient square metric windows centered on each unit:

- ±100 m;
- ±250 m;
- ±500 m.

These are not circular-radius statistics.

## Next step

If these audits are acceptable, freeze terrain V1 and proceed to:

`04D0_LANDFIRE_pre_event_vintage_audit.ipynb`

before any fuel features are extracted.
"""

REPORT_PATH = (
    OUTPUT_ROOT
    / "TERRAIN_PRODUCTION_FEATURE_REPORT.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print()
print(
    "Saved:",
    REPORT_PATH
)

## 18. Output inventory

In [ ]:
print("=" * 96)
print("04C1 OUTPUTS")
print("=" * 96)

print(
    "Terrain feature table:"
)

print(
    FEATURES_PATH
)

print()
print(
    "Report:"
)

print(
    REPORT_PATH
)

print()
print(
    "Output files:"
)

for path in sorted(
    OUTPUT_ROOT.glob(
        "*"
    )
):

    if path.is_file():

        print(
            f"{path.name:<70}"
            f"{path.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "Block feature shards:",
    len(
        list(
            BLOCK_SHARD_ROOT.glob(
                "*.parquet"
            )
        )
    ),
)

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID
)